# Drug Review Sentiment Classification
## Complete Pipeline: EDA → Training → Multi-Model Evaluation

This notebook is the **single source of truth** for the full project. It covers:

1. **EDA** — Exploratory analysis of the raw Drugs.com dataset
2. **Preprocessing** — Filtering, ternary labelling, undersampling, stratified split
3. **Model Architecture** — Transformer encoder + custom classification head
4. **Training** — Fine-tuning with AdamW, differential LRs, warmup scheduler, gradient clipping
5. **Per-model evaluation** — Loss trajectory, epoch dynamics, generalisation gap, per-class metrics
6. **Multi-model comparison** — Cross-model validation, overall performance, heatmaps, radar chart

**Models evaluated:**

| Model | HuggingFace ID | Role |
|---|---|---|
| RoBERTa-base | `roberta-base` | General NLP baseline |
| Bio-ClinicalBERT | `emilyalsentzer/Bio_ClinicalBERT` | Biomedical domain |
| DeBERTa v3 | `microsoft/deberta-v3-base` | Architectural improvements |
| Sentiment-RoBERTa-Large | `siebert/sentiment-roberta-large-english` | Sentiment specialist |

**Training config:** 3 epochs · lr = 2e-5 (encoder) / 2e-4 (head) · batch = 64 · max_len = 256  
**Dataset:** drugsCOM_balanced.csv · 80/20 stratified split


---
## 1. Imports and Configuration

In [ ]:
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from transformers import AutoModel, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, accuracy_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("All imports loaded successfully.")


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
RAW_DATA_PATH      = Path("dataset/drugsCOM_raw.csv")
BALANCED_DATA_PATH = Path("dataset/drugsCOM_balanced.csv")

# ── Labels ─────────────────────────────────────────────────────────────────
LABEL_MAP   = {"bad": 0, "medium": 1, "good": 2}
LABEL_NAMES = ["bad", "medium", "good"]

# ── Hyperparameters ────────────────────────────────────────────────────────
SEED          = 42
EPOCHS        = 3
BATCH_SIZE    = 64
LR            = 2e-5
MAX_LEN       = 256
DROPOUT       = 0.1
HEAD_LR_MULT  = 10
WARMUP_FRAC   = 0.1
MAX_GRAD_NORM = 1.0
TEST_SIZE     = 0.2

# ── Multi-model config (used in Section 7) ─────────────────────────────────
MODEL_DIRS = {
    "Bio-ClinicalBERT":        "Bio_ClinicalBERT",
    "RoBERTa-Base":            "roberta-base",
    "Sentiment-RoBERTa-Large": "sentiment-roberta-large-english",
}
COLORS = {
    "Bio-ClinicalBERT":        "#4C72B0",
    "RoBERTa-Base":            "#E07B39",
    "Sentiment-RoBERTa-Large": "#2E9E5B",
}
MARKERS = {
    "Bio-ClinicalBERT":        "o",
    "RoBERTa-Base":            "s",
    "Sentiment-RoBERTa-Large": "^",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ── Reproducibility ────────────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Random seed set to {SEED}")


---
## 2. Exploratory Data Analysis (EDA)

We analyse the raw Drugs.com dataset before any preprocessing to understand its structure,
the distribution of ratings, and the characteristics of the review texts.


In [ ]:
# Load raw dataset
df = pd.read_csv(RAW_DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nNull values per column:")
print(df.isnull().sum())
df.head()


In [ ]:
# ── Review length statistics ───────────────────────────────────────────────
review_lengths = df["review"].apply(lambda x: len(str(x).split()))
print(f"Average length:      {review_lengths.mean():.2f} words")
print(f"Median length:       {review_lengths.median():.2f} words")
print(f"95th percentile:     {review_lengths.quantile(0.95):.2f} words")
print(f"Max length:          {review_lengths.max()} words")
print(f"\nReviews <= 256 words: {(review_lengths <= 256).sum():,} ({(review_lengths <= 256).mean():.1%})")


In [ ]:
# ── Rating distribution (bar chart) ───────────────────────────────────────
rating_counts = df["rating"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Raw Dataset — Rating Distribution", fontsize=13, fontweight="bold")

# Bar chart
axes[0].bar(rating_counts.index, rating_counts.values, color="steelblue", edgecolor="white", linewidth=0.8)
axes[0].set_xlabel("Rating (1-10)")
axes[0].set_ylabel("Number of reviews")
axes[0].set_title("Rating counts", fontweight="bold")
axes[0].set_xticks(rating_counts.index)

# Pie chart with ternary buckets (preview)
bad_count    = (df["rating"] < 4).sum()
medium_count = ((df["rating"] >= 4) & (df["rating"] <= 7)).sum()
good_count   = (df["rating"] > 7).sum()
axes[1].pie([bad_count, medium_count, good_count],
            labels=["bad (<4)", "medium (4-7)", "good (>7)"],
            colors=["#C0392B", "#F39C12", "#27AE60"],
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[1].set_title("Ternary class distribution (before balancing)", fontweight="bold")

plt.tight_layout()
plt.savefig("fig_eda_rating_dist.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# ── Review length distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(review_lengths, bins=80, color="steelblue", edgecolor="white", linewidth=0.5, alpha=0.85)
ax.axvline(256, color="crimson", linestyle="--", linewidth=2, label="Truncation threshold (256)")
ax.axvline(review_lengths.quantile(0.95), color="orange", linestyle="--", linewidth=1.5,
           label=f"P95 = {review_lengths.quantile(0.95):.0f}")
ax.set_xlabel("Review length (words)")
ax.set_ylabel("Count")
ax.set_title("Review length distribution", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("fig_eda_length_dist.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# ── Top drugs and conditions ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Most reviewed drugs and conditions", fontsize=13, fontweight="bold")

top_drugs = df["drugName"].value_counts().head(15)
axes[0].barh(top_drugs.index[::-1], top_drugs.values[::-1], color="steelblue", alpha=0.85)
axes[0].set_xlabel("Number of reviews")
axes[0].set_title("Top 15 drugs", fontweight="bold")

top_cond = df["condition"].value_counts().head(15)
axes[1].barh(top_cond.index[::-1], top_cond.values[::-1], color="#E07B39", alpha=0.85)
axes[1].set_xlabel("Number of reviews")
axes[1].set_title("Top 15 conditions", fontweight="bold")

plt.tight_layout()
plt.savefig("fig_eda_top_drugs_conditions.png", bbox_inches="tight", dpi=150)
plt.show()


---
## 3. Preprocessing

1. Filter reviews longer than 256 words (transformer token limit)
2. Bucket ratings 1–10 into three sentiment classes
3. Undersample to balance classes
4. Stratified 80/20 train/test split


In [ ]:
# ── Filter by length ───────────────────────────────────────────────────────
df["review_length"] = df["review"].apply(lambda x: len(str(x).split()))
df_filtered = df[df["review_length"] <= MAX_LEN].copy()

print(f"Original samples:          {len(df):,}")
print(f"After filtering (<={MAX_LEN} words): {len(df_filtered):,}")
print(f"Removed:                   {len(df) - len(df_filtered):,} ({(len(df) - len(df_filtered)) / len(df):.1%})")


In [ ]:
# ── Ternary labelling ──────────────────────────────────────────────────────
# bad: rating < 4  |  medium: 4 <= rating <= 7  |  good: rating > 7
bins   = [0, 4, 8, 11]
labels = ["bad", "medium", "good"]
df_filtered["label"] = pd.cut(df_filtered["rating"], bins=bins, labels=labels,
                               right=False, include_lowest=True)

counts    = df_filtered["label"].value_counts()
min_count = counts.min()

print("Label distribution before balancing:")
for lbl in labels:
    print(f"  {lbl:>8}: {counts[lbl]:>7,}  ({counts[lbl] / len(df_filtered):.1%})")
print(f"\nMinority class size: {min_count:,}")


In [ ]:
# ── Undersampling ──────────────────────────────────────────────────────────
df_balanced = pd.concat([
    group.sample(n=min_count, random_state=SEED)
    for _, group in df_filtered.groupby("label", observed=True)
]).reset_index(drop=True)

df_balanced.to_csv(BALANCED_DATA_PATH, index=False)

print("Balanced class distribution:")
print(df_balanced["label"].value_counts())
print(f"\nTotal balanced samples: {len(df_balanced):,}")
print(f"Saved to: {BALANCED_DATA_PATH}")


In [ ]:
# ── Stratified 80/20 split ─────────────────────────────────────────────────
X = df_balanced["review"]
y = df_balanced["label"].map(LABEL_MAP)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print(f"Training set:  {len(X_train):,} samples")
print(f"Test set:      {len(X_test):,} samples")
print("\nClass proportions in training set:")
print(y_train.value_counts(normalize=True).round(4))
print("\nClass proportions in test set:")
print(y_test.value_counts(normalize=True).round(4))


---
## 4. Model Architecture

Each model consists of a **pretrained Transformer encoder** (frozen pre-trained weights, fine-tuned
end-to-end) plus a **trainable classification head**:

```
Input text → Tokenizer → Encoder → Mean Pooling → Linear(hidden→256) → ReLU → Dropout → Linear(256→3) → Logits
```


In [ ]:
class DrugReviewClassifier(nn.Module):
    def __init__(self, model_name: str, num_labels: int = 3, dropout: float = 0.1):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size   # 768 (base) or 1024 (large)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def mean_pooling(self, token_embeddings, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled  = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        return self.classifier(pooled)


class DrugReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

print("Classes defined.")


---
## 5. Training Pipeline

### Training and evaluation functions


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, batch_losses, all_preds, all_labels = 0, [], [], []

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        loss_val = loss.item()
        total_loss += loss_val
        batch_losses.append(loss_val)
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return batch_losses, avg_loss, macro_f1


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)

            total_loss += loss.item()
            all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, macro_f1, all_preds, all_labels

print("train_epoch and eval_epoch defined.")


### Main training loop

Set `MODEL_NAME` to the desired model before running this cell.  
Results are saved to `models/<MODEL_NAME>/` (epoch_metrics.csv, batch_losses.csv, best_model.pt, classification_report.txt).


In [ ]:
# ── Select model to train ──────────────────────────────────────────────────
MODEL_NAME = "roberta-base"   # change to any of the four models
# MODEL_NAME = "Bio_ClinicalBERT"
# MODEL_NAME = "deberta-v3-base"
# MODEL_NAME = "sentiment-roberta-large-english"

OUTPUT_DIR = Path("models") / MODEL_NAME.replace("/", "_")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load tokenizer and model ───────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = DrugReviewClassifier(MODEL_NAME, num_labels=3, dropout=DROPOUT).to(device)

n_params    = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME}")
print(f"Total parameters:     {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

# ── Datasets and loaders ───────────────────────────────────────────────────
train_dataset = DrugReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset  = DrugReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches/epoch: {len(train_loader)} | Test batches: {len(test_loader)}")

# ── Optimiser and scheduler ────────────────────────────────────────────────
criterion    = nn.CrossEntropyLoss()
optimizer    = AdamW(
    [
        {"params": model.encoder.parameters(),    "lr": LR},
        {"params": model.classifier.parameters(), "lr": LR * HEAD_LR_MULT},
    ],
    weight_decay=0.01,
)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_FRAC * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"Total steps: {total_steps} | Warmup: {warmup_steps} | Encoder LR: {LR} | Head LR: {LR * HEAD_LR_MULT}")

# ── Save config ────────────────────────────────────────────────────────────
config = {
    "model_name": MODEL_NAME, "seed": SEED, "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "lr": LR, "max_len": MAX_LEN, "dropout": DROPOUT,
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

# ── Training loop ──────────────────────────────────────────────────────────
best_f1, best_state          = 0, None
epoch_rows, batch_rows       = [], []
global_step                  = 0

for epoch in range(1, EPOCHS + 1):
    train_batch_losses, train_loss, train_f1 = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )
    val_loss, val_f1, preds, gt = eval_epoch(model, test_loader, criterion, device)

    for bl in train_batch_losses:
        global_step += 1
        batch_rows.append({"step": global_step, "epoch": epoch, "loss": bl})

    epoch_rows.append({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "train_f1": train_f1, "val_f1": val_f1, "end_step": global_step,
    })

    pd.DataFrame(epoch_rows).to_csv(OUTPUT_DIR / "epoch_metrics.csv", index=False)
    pd.DataFrame(batch_rows).to_csv(OUTPUT_DIR / "batch_losses.csv",  index=False)

    print(f"Epoch {epoch}/{EPOCHS} | Train loss: {train_loss:.4f}  F1: {train_f1:.4f} | "
          f"Val loss: {val_loss:.4f}  F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1  = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  New best model (F1={best_f1:.4f})")

# ── Save best model ────────────────────────────────────────────────────────
model.load_state_dict(best_state)
torch.save(best_state, OUTPUT_DIR / "best_model.pt")
print(f"\nBest model saved → {OUTPUT_DIR / 'best_model.pt'}")

# ── Final evaluation ───────────────────────────────────────────────────────
_, _, preds, gt = eval_epoch(model, test_loader, criterion, device)
report = classification_report(gt, preds, target_names=LABEL_NAMES)
(OUTPUT_DIR / "classification_report.txt").write_text(report)
print("\nFinal Classification Report:")
print(report)

epoch_df = pd.DataFrame(epoch_rows)
batch_df = pd.DataFrame(batch_rows)


---
## 6. Per-Model Evaluation Visualisations

Run this section after training to inspect the behaviour of the **currently loaded model**.


### 6.1 Training Loss Trajectory (Batch Level)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

steps    = batch_df["step"].values
losses   = batch_df["loss"].values.astype(float)
smoothed = pd.Series(losses).rolling(window=100, center=True, min_periods=1).mean().values

ax.plot(steps, losses,   color="steelblue", alpha=0.15, linewidth=0.7)
ax.plot(steps, smoothed, color="steelblue", linewidth=2.5, label="Smoothed (w=100)")

boundaries = [0] + list(epoch_df["end_step"].values)
for end in epoch_df["end_step"].values[:-1]:
    ax.axvline(end, color="gray", linestyle="--", linewidth=1.0, alpha=0.7)
for i in range(EPOCHS):
    mid = (boundaries[i] + boundaries[i + 1]) / 2
    ax.text(mid, ax.get_ylim()[0] + 0.02, f"Epoch {i + 1}", ha="center", fontsize=10,
            color="#555", bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"))

ax.set_xlabel("Training Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title(f"Training Loss Trajectory — {MODEL_NAME}", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_batch_loss.png", bbox_inches="tight", dpi=150)
plt.show()


### 6.2 Epoch-Level Training Dynamics

In [ ]:
fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(14, 4.5))

epochs_arr = epoch_df["epoch"].values

ax_loss.plot(epochs_arr, epoch_df["train_loss"], "o--", color="steelblue", lw=2, ms=8, label="Train loss")
ax_loss.plot(epochs_arr, epoch_df["val_loss"],   "s-",  color="steelblue", lw=2, ms=8, alpha=0.55, label="Val loss")
ax_loss.fill_between(epochs_arr, epoch_df["train_loss"], epoch_df["val_loss"], color="steelblue", alpha=0.08)
ax_loss.set(xlabel="Epoch", ylabel="Cross-Entropy Loss", title="Loss per Epoch")
ax_loss.set_xticks(range(1, EPOCHS + 1))
ax_loss.legend(fontsize=9)

ax_f1.plot(epochs_arr, epoch_df["train_f1"], "o--", color="steelblue", lw=2, ms=8, label="Train F1")
ax_f1.plot(epochs_arr, epoch_df["val_f1"],   "s-",  color="steelblue", lw=2, ms=8, alpha=0.55, label="Val F1")
ax_f1.fill_between(epochs_arr, epoch_df["train_f1"], epoch_df["val_f1"], color="steelblue", alpha=0.08)
ax_f1.set(xlabel="Epoch", ylabel="Macro F1", title="Macro F1 per Epoch")
ax_f1.set_xticks(range(1, EPOCHS + 1))
ax_f1.legend(fontsize=9)

fig.suptitle(f"Epoch-Level Training Dynamics — {MODEL_NAME}", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_epoch_dynamics.png", bbox_inches="tight", dpi=150)
plt.show()


### 6.3 Generalisation Gap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
gap = epoch_df["train_f1"] - epoch_df["val_f1"]
ax.plot(epoch_df["epoch"], gap, "o-", color="crimson", lw=2.5, ms=10)
ax.axhline(0, color="gray", linestyle="--", lw=1.0, alpha=0.7)
ax.fill_between(epoch_df["epoch"], 0, gap, color="crimson", alpha=0.1)
ax.set(xlabel="Epoch", ylabel="Train F1 − Val F1", title="Generalisation Gap")
ax.set_xticks(range(1, EPOCHS + 1))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_generalisation.png", bbox_inches="tight", dpi=150)
plt.show()


### 6.4 Per-Class Performance

In [ ]:
report_dict = classification_report(gt, preds, target_names=LABEL_NAMES, output_dict=True)
class_data  = {cls: report_dict[cls] for cls in LABEL_NAMES}
accuracy    = report_dict["accuracy"]
macro_f1    = report_dict["macro avg"]["f1-score"]

print(f"Test Accuracy: {accuracy:.3f}  |  Macro F1: {macro_f1:.3f}\n")
display(pd.DataFrame(class_data).T.round(3))


In [ ]:
# Heatmap: precision / recall / F1 per class
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, metric in zip(axes, ["precision", "recall", "f1-score"]):
    mat = pd.DataFrame({cls: [class_data[cls][metric]] for cls in LABEL_NAMES}, index=[MODEL_NAME])
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="YlGnBu",
                vmin=0.60, vmax=0.92, linewidths=0.6, linecolor="white",
                annot_kws={"size": 14, "weight": "bold"},
                ax=ax, cbar=(metric == "f1-score"))
    ax.set_title(metric.capitalize(), fontsize=12, fontweight="bold")
    ax.set_xlabel("Class")
fig.suptitle(f"Per-Class Performance — {MODEL_NAME}", fontsize=13, fontweight="bold", y=1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_heatmap.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# Grouped bar: F1 per class
class_colors = {"bad": "#C0392B", "medium": "#F39C12", "good": "#27AE60"}
fig, ax = plt.subplots(figsize=(8, 5))
f1_vals = [class_data[cls]["f1-score"] for cls in LABEL_NAMES]
bars    = ax.bar(LABEL_NAMES, f1_vals,
                 color=[class_colors[cls] for cls in LABEL_NAMES],
                 alpha=0.82, edgecolor="white", linewidth=1.2, width=0.5)
for bar, v in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
            f"{v:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set(ylabel="F1 Score", title=f"Per-Class F1 — {MODEL_NAME}", ylim=(0, 1.05))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_perclass_f1.png", bbox_inches="tight", dpi=150)
plt.show()


### 6.5 Summary Table (single model)

In [ ]:
summary = {
    "Model":          MODEL_NAME,
    "Test Accuracy":  round(accuracy, 4),
    "Macro F1":       round(macro_f1, 4),
}
for cls in LABEL_NAMES:
    summary[f"F1 — {cls.capitalize()}"] = round(class_data[cls]["f1-score"], 4)
summary["Best Val F1"]    = round(epoch_df["val_f1"].max(), 4)
summary["Final Train F1"] = round(epoch_df["train_f1"].iloc[-1], 4)

summary_df = pd.DataFrame([summary]).set_index("Model")
display(summary_df.style.format("{:.4f}").set_caption("Test Set Performance Summary"))


---
## 7. Multi-Model Comparison

Load pre-saved metrics from all trained models and compare them visually.  
**Prerequisite:** all four models must have been trained and their results saved to `models/<model_name>/`.


In [ ]:
# ── Load metrics for all models ────────────────────────────────────────────
epoch_dfs = {}
batch_dfs = {}
for label, folder in MODEL_DIRS.items():
    p = Path("models") / folder
    epoch_dfs[label] = pd.read_csv(p / "epoch_metrics.csv")
    batch_dfs[label] = pd.read_csv(p / "batch_losses.csv")

# ── Classification report values (from classification_report.txt) ──────────
# Update these values after training all models
clf_report = {
    "Bio-ClinicalBERT": {
        "bad":    {"Precision": 0.83, "Recall": 0.82, "F1": 0.82},
        "medium": {"Precision": 0.68, "Recall": 0.74, "F1": 0.71},
        "good":   {"Precision": 0.87, "Recall": 0.81, "F1": 0.84},
        "Accuracy": 0.79, "Macro F1": 0.79,
    },
    "RoBERTa-Base": {
        "bad":    {"Precision": 0.83, "Recall": 0.85, "F1": 0.84},
        "medium": {"Precision": 0.72, "Recall": 0.72, "F1": 0.72},
        "good":   {"Precision": 0.87, "Recall": 0.85, "F1": 0.86},
        "Accuracy": 0.81, "Macro F1": 0.81,
    },
    "Sentiment-RoBERTa-Large": {
        "bad":    {"Precision": 0.87, "Recall": 0.84, "F1": 0.86},
        "medium": {"Precision": 0.73, "Recall": 0.77, "F1": 0.75},
        "good":   {"Precision": 0.88, "Recall": 0.86, "F1": 0.87},
        "Accuracy": 0.82, "Macro F1": 0.83,
    },
}
models = list(clf_report.keys())
CLASSES = ["bad", "medium", "good"]

print("Multi-model data loaded.")
for label, df in epoch_dfs.items():
    print(f"  {label}: {len(df)} epochs | {len(batch_dfs[label]):,} training steps")


### 7.1 Training Loss Trajectory — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle("Training Loss Trajectory (Batch Level) — All Models",
             fontsize=14, fontweight="bold", y=1.01)

for label, df in batch_dfs.items():
    color    = COLORS[label]
    steps    = df["step"].values
    losses   = df["loss"].values.astype(float)
    smoothed = pd.Series(losses).rolling(window=100, center=True, min_periods=1).mean().values
    ax.plot(steps, losses,   color=color, alpha=0.10, linewidth=0.6)
    ax.plot(steps, smoothed, color=color, linewidth=2, label=f"{label} (smoothed, w=100)")

ax.set(xlabel="Training Step", ylabel="Cross-Entropy Loss")
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig("fig1_batch_loss.png", bbox_inches="tight", dpi=150)
plt.show()


### 7.2 Epoch-Level Training Dynamics — All Models

In [ ]:
fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Epoch-Level Training Dynamics — All Models", fontsize=14, fontweight="bold", y=1.01)

for label, df in epoch_dfs.items():
    c, m = COLORS[label], MARKERS[label]
    ax_loss.plot(df["epoch"], df["train_loss"], m + "--", color=c, lw=2, ms=8, label=f"{label} Train")
    ax_loss.plot(df["epoch"], df["val_loss"],   m + "-",  color=c, lw=2, ms=8, alpha=0.55, label=f"{label} Val")
    ax_f1.plot(  df["epoch"], df["train_f1"],   m + "--", color=c, lw=2, ms=8, label=f"{label} Train")
    ax_f1.plot(  df["epoch"], df["val_f1"],     m + "-",  color=c, lw=2, ms=8, alpha=0.55, label=f"{label} Val")

for ax, ylabel, title in [
    (ax_loss, "Cross-Entropy Loss", "Loss per Epoch"),
    (ax_f1,   "Macro F1",          "Macro F1 per Epoch"),
]:
    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.set_xticks([1, 2, 3])
    ax.legend(fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.savefig("fig2_epoch_dynamics.png", bbox_inches="tight", dpi=150)
plt.show()


### 7.3 Cross-Model Validation Comparison

In [ ]:
fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Cross-Model Validation Metrics", fontsize=14, fontweight="bold")

for label, df in epoch_dfs.items():
    kw = dict(color=COLORS[label], linewidth=2.2, markersize=10, label=label)
    ax_loss.plot(df["epoch"], df["val_loss"], MARKERS[label] + "-", **kw)
    ax_f1.plot(  df["epoch"], df["val_f1"],  MARKERS[label] + "-", **kw)

for ax, ylabel, title in [
    (ax_loss, "Validation Loss",     "Validation Loss per Epoch"),
    (ax_f1,   "Validation Macro F1", "Validation F1 per Epoch"),
]:
    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.set_xticks([1, 2, 3])
    ax.legend(fontsize=9.5, framealpha=0.9)

plt.tight_layout()
plt.savefig("fig3_cross_model_val.png", bbox_inches="tight", dpi=150)
plt.show()


### 7.4 Overall Test Set Performance

In [ ]:
accuracy_vals = [clf_report[m]["Accuracy"]  for m in models]
macro_f1_vals = [clf_report[m]["Macro F1"]  for m in models]
x = np.arange(len(models))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5.5))
bars1 = ax.bar(x - w / 2, accuracy_vals, w, label="Accuracy",
               color=[COLORS[m] for m in models], alpha=0.90, edgecolor="white", lw=1.2)
bars2 = ax.bar(x + w / 2, macro_f1_vals, w, label="Macro F1",
               color=[COLORS[m] for m in models], alpha=0.50, edgecolor="white", lw=1.2, hatch="//")

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.003,
            f"{h:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0.70, 0.89)
ax.set(ylabel="Score", title="Test Set Accuracy and Macro F1 by Model")
ax.legend(fontsize=10, framealpha=0.9)
ax.axhline(0.80, color="red", linestyle=":", lw=1.2, alpha=0.45)
ax.text(2.5, 0.802, "0.80 reference", fontsize=8.5, color="red", alpha=0.7, ha="right")

best_idx = int(np.argmax(macro_f1_vals))
ax.annotate("Best\nmodel",
            xy=(x[best_idx] + w / 2, macro_f1_vals[best_idx] + 0.003),
            xytext=(x[best_idx] + w / 2 + 0.35, macro_f1_vals[best_idx] + 0.020),
            arrowprops=dict(arrowstyle="->", color="#2E9E5B", lw=1.8),
            fontsize=9.5, color="#2E9E5B", fontweight="bold")

plt.tight_layout()
plt.savefig("fig4_overall_performance.png", bbox_inches="tight", dpi=150)
plt.show()


### 7.5 Per-Class Performance — Heatmaps and Bar Chart

In [ ]:
# Heatmaps
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
fig.suptitle("Per-Class Precision, Recall and F1 — All Models", fontsize=14, fontweight="bold")

for ax, metric in zip(axes, ["Precision", "Recall", "F1"]):
    mat = pd.DataFrame(
        {cls: [clf_report[m][cls][metric] for m in models] for cls in CLASSES},
        index=models,
    )
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="YlGnBu",
                vmin=0.65, vmax=0.92, linewidths=0.6, linecolor="white",
                annot_kws={"size": 13, "weight": "bold"},
                ax=ax, cbar=(metric == "F1"))
    ax.set_title(metric, fontsize=13, fontweight="bold")
    ax.set_xlabel("Class", fontsize=10)
    ax.set_ylabel("Model" if metric == "Precision" else "", fontsize=10)
    ax.tick_params(axis="x", labelsize=10)
    ax.tick_params(axis="y", labelsize=9, rotation=0)

plt.tight_layout()
plt.savefig("fig5_heatmaps.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# Grouped bar: per-class F1
class_colors = {"bad": "#C0392B", "medium": "#F39C12", "good": "#27AE60"}
n_classes  = len(CLASSES)
bar_width  = 0.22
x          = np.arange(len(models))

fig, ax = plt.subplots(figsize=(12, 5.5))
for i, cls in enumerate(CLASSES):
    offset   = (i - (n_classes - 1) / 2) * bar_width
    f1_vals  = [clf_report[m][cls]["F1"] for m in models]
    bars     = ax.bar(x + offset, f1_vals, bar_width,
                      label=cls.capitalize(), color=class_colors[cls],
                      alpha=0.82, edgecolor="white", lw=0.8)
    for bar, v in zip(bars, f1_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.003,
                f"{v:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0.58, 0.95)
ax.set(ylabel="F1 Score", title="Per-Class F1 Score by Model")
ax.legend(title="Sentiment Class", fontsize=10, title_fontsize=10, framealpha=0.9)
ax.axhline(0.80, color="gray", linestyle=":", lw=1.0, alpha=0.6)

plt.tight_layout()
plt.savefig("fig6_perclass_f1.png", bbox_inches="tight", dpi=150)
plt.show()


### 7.6 Generalisation Analysis — All Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Generalisation Analysis", fontsize=14, fontweight="bold")

ax_left, ax_right = axes

for label, df in epoch_dfs.items():
    c = COLORS[label]
    ax_left.plot(df["epoch"], df["train_f1"], "--", color=c, lw=1.8, alpha=0.9)
    ax_left.plot(df["epoch"], df["val_f1"],   "-",  color=c, lw=2.2,
                 marker=MARKERS[label], ms=9, label=label)
    ax_left.fill_between(df["epoch"], df["train_f1"], df["val_f1"], color=c, alpha=0.07)

    gap = (df["train_f1"] - df["val_f1"]).values
    ax_right.plot(df["epoch"], gap, MARKERS[label] + "-", color=c, lw=2.2, ms=9, label=label)

ax_left.set(title="Train (dashed) vs Val (solid) F1", xlabel="Epoch", ylabel="Macro F1")
ax_left.set_xticks([1, 2, 3])
ax_left.legend(fontsize=9, framealpha=0.9)

ax_right.axhline(0, color="gray", linestyle="--", lw=1.0, alpha=0.7)
ax_right.fill_between([0.9, 3.1], 0, 0.12, color="#FFCCCC", alpha=0.15, label="Overfitting zone")
ax_right.set(title="Train − Val F1 Gap", xlabel="Epoch", ylabel="Train F1 − Val F1")
ax_right.set_xticks([1, 2, 3])
ax_right.set_xlim(0.8, 3.2)
ax_right.legend(fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig("fig7_generalisation.png", bbox_inches="tight", dpi=150)
plt.show()


### 7.7 Training Loss vs Val F1 Scatter (Tradeoff)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for label, df in epoch_dfs.items():
    final_loss = df["train_loss"].iloc[-1]
    best_f1    = df["val_f1"].max()
    test_acc   = clf_report[label]["Accuracy"]

    ax.scatter(final_loss, best_f1,
               s=test_acc * 1200, color=COLORS[label], alpha=0.75,
               edgecolors="white", linewidth=2, zorder=3, label=label)
    ax.annotate(
        f"{label}\n(acc={test_acc:.0%})",
        xy=(final_loss, best_f1),
        xytext=(final_loss + 0.004, best_f1 + 0.0015),
        fontsize=9, ha="left",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8, ec="gray"),
    )

ax.set(xlabel="Final Training Loss (Epoch 3)", ylabel="Best Validation F1",
       title="Training Loss vs Best Val F1\n(bubble size ∝ test accuracy)")
ax.text(0.02, 0.98, "Better →", transform=ax.transAxes,
        ha="left", va="top", fontsize=9, color="gray", style="italic")

plt.tight_layout()
plt.savefig("fig_scatter_tradeoff.png", bbox_inches="tight", dpi=150)
plt.show()


### 7.8 Radar Chart — Multi-Dimensional Performance

In [ ]:
categories = ["Accuracy", "Macro F1", "F1 — Bad", "F1 — Medium", "F1 — Good"]
N      = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0.60, 0.92)
ax.set_yticks([0.65, 0.70, 0.75, 0.80, 0.85, 0.90])
ax.set_yticklabels(["0.65", "0.70", "0.75", "0.80", "0.85", "0.90"], fontsize=7.5, color="gray")
ax.grid(color="gray", linestyle="--", lw=0.5, alpha=0.5)

for label, data in clf_report.items():
    values = [data["Accuracy"], data["Macro F1"],
              data["bad"]["F1"], data["medium"]["F1"], data["good"]["F1"]]
    values += values[:1]
    ax.plot(angles, values, "o-", lw=2.3, color=COLORS[label], label=label, markersize=7)
    ax.fill(angles, values, alpha=0.10, color=COLORS[label])

ax.legend(loc="upper left", bbox_to_anchor=(-0.35, 1.15), fontsize=10, framealpha=0.9)
ax.set_title("Model Performance Radar", fontsize=14, fontweight="bold", pad=28)

plt.tight_layout()
plt.savefig("fig8_radar.png", bbox_inches="tight", dpi=150)
plt.show()


### 7.9 Summary Table — All Models

In [ ]:
rows = []
for m in models:
    d = clf_report[m]
    rows.append({
        "Model":          m,
        "Test Accuracy":  d["Accuracy"],
        "Macro F1":       d["Macro F1"],
        "Bad F1":         d["bad"]["F1"],
        "Medium F1":      d["medium"]["F1"],
        "Good F1":        d["good"]["F1"],
        "Best Val F1":    round(epoch_dfs[m]["val_f1"].max(), 4),
        "Final Train F1": round(epoch_dfs[m]["train_f1"].iloc[-1], 4),
    })

summary_df = pd.DataFrame(rows).set_index("Model")
styled = (
    summary_df.style
    .set_caption("Test Set Performance Summary — All Models")
    .format("{:.3f}")
    .set_table_styles([
        {"selector": "caption", "props": "caption-side: top; font-size: 1.2em; font-weight: bold; padding-bottom: 8px;"},
        {"selector": "th",      "props": "background-color: #2C3E50; color: white; padding: 9px 14px; font-size: 11px;"},
        {"selector": "td",      "props": "padding: 8px 14px; text-align: center; font-size: 11px;"},
        {"selector": "tr:nth-child(even)", "props": "background-color: #F2F2F2;"},
    ])
    .highlight_max(axis=0, color="#ABEBC6")
    .highlight_min(axis=0, color="#FADBD8")
)
display(styled)


---
## 8. Conclusions

### Model Ranking
1. **Sentiment-RoBERTa-Large** — Best overall (82% accuracy, 0.83 macro F1).  
   Sentiment-focused pre-training aligns directly with the task; larger capacity (~355M params) also contributes.
2. **RoBERTa-Base** — Strong runner-up (81% / 0.81).  
   Robustly optimised pre-training beats domain adaptation on this opinion-heavy dataset.
3. **Bio-ClinicalBERT** — Weakest overall (79% / 0.79).  
   Clinical text is factual, not opinion-driven — the domain transfer is a mismatch for consumer sentiment.

### Class Difficulty
| Class | Typical F1 range | Why |
|-------|-----------------|-----|
| **Medium** | 0.71 – 0.75 | Ambiguous reviews at the boundary of bad/good sentiment |
| **Bad** | 0.82 – 0.86 | Strong negative language is distinctive |
| **Good** | 0.84 – 0.87 | Strong positive language is most frequent and distinctive |

### Generalisation
All models show a mild train–val F1 gap that widens with epochs, but converging val losses suggest
training has not fully saturated. Additional epochs or a larger learning rate could further improve performance.

### Recommendation
- **Best performance:** `sentiment-roberta-large-english` — highest accuracy and F1 across all classes.
- **Cost-efficiency:** `roberta-base` — only ~2% lower macro F1 with 3× fewer parameters.
- **Avoid for this task:** `Bio_ClinicalBERT` — medical domain pre-training does not benefit consumer sentiment.
